# ST 597 — Chicago Traffic Crashes
Right now just grabs the data from a local folder, processes it, splits into training and test data, creates CV splits, sets up the data dictonary for the protocol checkpoint, and starts setting up the baseline

### 1. Download & Splitting
Load in the data from local folder and keep only necessary years and variables; this step is skipped if the data has already been split into a subset and saved in the proper filepath. This also creates the response variable `multiple_injuries`.

In [15]:
import pandas as pd
from pathlib import Path

# Set the project folder
BASE = Path.cwd()

# Set the file paths
file_path = BASE / "data/raw/Traffic_Crashes_-_Crashes_20261013.csv"
processed_folder = BASE / "data/processed"
processed_folder.mkdir(parents=True, exist_ok=True)

output_file = processed_folder / "chicago_crashes_2023_2025.csv"

# Check whether the processed dataset already exists
if output_file.exists():

    print("Processed dataset already exists. Skipping data preparation.")

    # Load the saved dataset
    data_filtered = pd.read_csv(
        output_file,
        parse_dates=["CRASH_DATE"],
        low_memory=False
    )

else:

    print("No processed dataset found. Starting data preparation.")

    # Load the original Chicago Traffic Crashes dataset
    data = pd.read_csv(file_path, low_memory=False)

    print("Original observations:", len(data))
    print("Original columns:", len(data.columns))

    # Convert crash dates to datetime
    data["CRASH_DATE"] = pd.to_datetime(
        data["CRASH_DATE"],
        format="%m/%d/%Y %I:%M:%S %p",
        errors="coerce"
    )

    # Keep crashes from 2023 through 2025
    data_filtered = data[
        (data["CRASH_DATE"] >= "2023-01-01") &
        (data["CRASH_DATE"] < "2026-01-01")
    ].copy()

    print("Observations from 2023-2025:", len(data_filtered))

    # Keep only the variables needed for the project
    columns_to_keep = [
        "CRASH_RECORD_ID",
        "CRASH_DATE",
        "STREET_NAME",
        "INJURIES_TOTAL",
        "POSTED_SPEED_LIMIT",
        "WEATHER_CONDITION",
        "LIGHTING_CONDITION",
        "ROADWAY_SURFACE_COND",
        "TRAFFIC_CONTROL_DEVICE",
        "ALIGNMENT",
        "ROAD_DEFECT",
        "INTERSECTION_RELATED_I",
        "STREET_DIRECTION",
        "TRAFFICWAY_TYPE",
        "CRASH_HOUR",
        "CRASH_DAY_OF_WEEK",
        "CRASH_MONTH",
        "WORK_ZONE_I",
        "WORK_ZONE_TYPE",
        "WORKERS_PRESENT_I"
    ]

    # Check that all required columns are present
    missing_columns = [
        col for col in columns_to_keep
        if col not in data_filtered.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")

    data_filtered = data_filtered[columns_to_keep].copy()

    # Convert injury counts to numeric
    data_filtered["INJURIES_TOTAL"] = pd.to_numeric(
        data_filtered["INJURIES_TOTAL"],
        errors="coerce"
    )

    # Remove records with invalid injury counts
    data_filtered = data_filtered[
        data_filtered["INJURIES_TOTAL"].notna() &
        (data_filtered["INJURIES_TOTAL"] >= 0) &
        (data_filtered["INJURIES_TOTAL"] % 1 == 0)
    ].copy()

    # Create the binary response variable
    data_filtered["multiple_injuries"] = (
        data_filtered["INJURIES_TOTAL"] >= 2
    ).astype(int)

    # Save the processed dataset
    data_filtered.to_csv(output_file, index=False)

    print("Processed dataset saved to:", output_file)

# These lines run whether the data was loaded or newly prepared
print("\nFinal observations:", len(data_filtered))
print("\nResponse distribution:")
print(data_filtered["multiple_injuries"].value_counts())

print(
    "\nPositive rate:",
    round(data_filtered["multiple_injuries"].mean() * 100, 2),
    "%"
)

display(data_filtered.head())

Processed dataset already exists. Skipping data preparation.

Final observations: 331203

Response distribution:
multiple_injuries
0    317496
1     13707
Name: count, dtype: int64

Positive rate: 4.14 %


,CRASH_RECORD_ID,CRASH_DATE,STREET_NAME,INJURIES_TOTAL,POSTED_SPEED_LIMIT,WEATHER_CONDITION,LIGHTING_CONDITION,ROADWAY_SURFACE_COND,TRAFFIC_CONTROL_DEVICE,ALIGNMENT,...,INTERSECTION_RELATED_I,STREET_DIRECTION,TRAFFICWAY_TYPE,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,WORK_ZONE_I,WORK_ZONE_TYPE,WORKERS_PRESENT_I,multiple_injuries
0,f3e70c341876e034264fd2b72e3851cb48419ac10e81b5...,2024-12-26 13:38:00,BAGGAGE LN,0.0,15,CLOUDY/OVERCAST,DAYLIGHT,WET,NO CONTROLS,STRAIGHT AND LEVEL,...,NaN,S,ONE-WAY,13,5,12,NaN,NaN,NaN,0
1,9f6c49a124ae031b10b0502712b89609bdeb12aaf5aeef...,2024-11-27 13:30:00,SPECIAL OLYMPICS DR,0.0,10,CLEAR,DAYLIGHT,DRY,NO CONTROLS,STRAIGHT AND LEVEL,...,NaN,S,PARKING LOT,13,4,11,NaN,NaN,NaN,0
2,8afb434835dc4244c636f8474a0bfe7eb0b9750ef885ab...,2024-12-14 22:48:00,MANNHEIM RD,0.0,30,RAIN,"DARKNESS, LIGHTED ROAD",WET,TRAFFIC SIGNAL,STRAIGHT AND LEVEL,...,Y,N,FOUR WAY,22,7,12,NaN,NaN,NaN,0
3,f0e5bd024c2d6368a219b5f539a113a5958009ef570174...,2024-12-08 23:01:00,BESSIE COLEMAN DR,0.0,30,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,TRAFFIC SIGNAL,STRAIGHT AND LEVEL,...,NaN,N,ONE-WAY,23,1,12,NaN,NaN,NaN,0
4,718fb2d17b518631d033536e1a978b527890f634218f85...,2024-11-30 16:30:00,WENTWORTH AVE,0.0,30,UNKNOWN,UNKNOWN,UNKNOWN,NO CONTROLS,STRAIGHT AND LEVEL,...,NaN,S,ONE-WAY,16,7,11,NaN,NaN,NaN,0


### 2. Training & Testing Split
Split the data into training (2023–2024) and testing (2025). Remove `INJURIES_TOTAL` to prevent data leakage and save both datasets; skip if files already exist.

In [ ]:
# Split the data into training and testing sets

# Set the file paths for the training and testing datasets
train_file = processed_folder / "train_2023_2024.csv"
test_file = processed_folder / "test_2025_DO_NOT_USE_FOR_TUNING.csv"

# Check whether the training and testing datasets already exist
if train_file.exists() and test_file.exists():

    print("Training and testing datasets already exist. Loading saved files.")

    train = pd.read_csv(
        train_file,
        parse_dates=["CRASH_DATE"],
        low_memory=False
    )

    test = pd.read_csv(
        test_file,
        parse_dates=["CRASH_DATE"],
        low_memory=False
    )

else:

    print("Creating training and testing datasets.")

    # Training data: crashes from 2023 and 2024
    train = data_filtered[
        data_filtered["CRASH_DATE"].dt.year.isin([2023, 2024])
    ].copy()

    # Testing data: crashes from 2025
    test = data_filtered[
        data_filtered["CRASH_DATE"].dt.year == 2025
    ].copy()

    # Remove the original injury count to prevent data leakage
    train = train.drop(columns=["INJURIES_TOTAL"])
    test = test.drop(columns=["INJURIES_TOTAL"])

    # Save the datasets
    train.to_csv(train_file, index=False)
    test.to_csv(test_file, index=False)

    print("Training and testing datasets saved.")

# Check the size of each dataset
print("\nTraining observations:", len(train))
print("Testing observations:", len(test))

# Check the response distribution in the training data
print("\nTraining response distribution:")
print(train["multiple_injuries"].value_counts())

print(
    "Training positive rate:",
    round(train["multiple_injuries"].mean() * 100, 2),
    "%"
)

# Check the response distribution in the testing data
print("\nTesting response distribution:")
print(test["multiple_injuries"].value_counts())

print(
    "Testing positive rate:",
    round(test["multiple_injuries"].mean() * 100, 2),
    "%"
)

# Verify the split is correct
assert train["CRASH_DATE"].dt.year.isin([2023, 2024]).all()
assert (test["CRASH_DATE"].dt.year == 2025).all()

# Make sure no crash appears in both datasets
assert set(train["CRASH_RECORD_ID"]).isdisjoint(
    set(test["CRASH_RECORD_ID"])
)

print("\nTraining/test split verified successfully!")

Training and testing datasets already exist. Loading saved files.

Training observations: 222301
Testing observations: 108902

Training response distribution:
multiple_injuries
0    213207
1      9094
Name: count, dtype: int64
Training positive rate: 4.09 %

Testing response distribution:
multiple_injuries
0    104289
1      4613
Name: count, dtype: int64
Testing positive rate: 4.24 %

Training/test split verified successfully!


### 3. Data Dictionary
Create a data dictionary with variable names, types, descriptions, and roles. Check missing values and unique values for each variable and save as a CSV.

In [17]:
# Create the data dictionary for the protocol checkpoint

# Define the variables used in the project
variable_info = {
    "CRASH_RECORD_ID": ("Identifier", "Categorical", "Unique identifier for each reported crash"),
    "CRASH_DATE": ("Split variable", "Datetime", "Date and time of the crash"),
    "STREET_NAME": ("CV grouping", "Categorical", "Street name used for grouped cross-validation"),
    "multiple_injuries": ("Response", "Binary", "1 if two or more people were injured; 0 otherwise"),

    "POSTED_SPEED_LIMIT": ("Predictor", "Numeric", "Posted speed limit at the crash location"),
    "WEATHER_CONDITION": ("Predictor", "Categorical", "Weather conditions at the time of the crash"),
    "LIGHTING_CONDITION": ("Predictor", "Categorical", "Lighting conditions at the time of the crash"),
    "ROADWAY_SURFACE_COND": ("Predictor", "Categorical", "Condition of the roadway surface"),
    "TRAFFIC_CONTROL_DEVICE": ("Predictor", "Categorical", "Type of traffic control device present"),
    "ALIGNMENT": ("Predictor", "Categorical", "Roadway alignment at the crash location"),
    "ROAD_DEFECT": ("Predictor", "Categorical", "Reported roadway defects"),
    "INTERSECTION_RELATED_I": ("Predictor", "Categorical", "Whether the crash was intersection-related"),
    "STREET_DIRECTION": ("Predictor", "Categorical", "Direction of the street"),
    "TRAFFICWAY_TYPE": ("Predictor", "Categorical", "Type of trafficway"),
    "CRASH_HOUR": ("Predictor", "Categorical", "Hour of day when the crash occurred"),
    "CRASH_DAY_OF_WEEK": ("Predictor", "Categorical", "Day of week when the crash occurred"),
    "CRASH_MONTH": ("Predictor", "Categorical", "Month when the crash occurred"),
    "WORK_ZONE_I": ("Predictor", "Categorical", "Whether the crash occurred in a work zone"),
    "WORK_ZONE_TYPE": ("Predictor", "Categorical", "Type of work zone"),
    "WORKERS_PRESENT_I": ("Predictor", "Categorical", "Whether workers were present in the work zone")
}

# Create a list to store information for each variable
dictionary_rows = []

for variable, (role, var_type, description) in variable_info.items():

    # Check that the variable exists in the training dataset
    if variable not in train.columns:
        raise ValueError(f"Variable missing from training data: {variable}")

    # Calculate missing values using training data only
    missing_count = train[variable].isna().sum()
    missing_percent = round(
        train[variable].isna().mean() * 100, 2
    )

    # Count unique nonmissing values
    unique_values = train[variable].nunique(dropna=True)

    # Define how each variable will be handled
    if role == "Predictor" and var_type == "Numeric":
        preparation = "Convert to numeric; median imputation within CV folds"

    elif role == "Predictor" and var_type == "Categorical":
        preparation = "Treat missing as Unknown; one-hot encode within CV folds"

    elif role == "Response":
        preparation = "Created from INJURIES_TOTAL >= 2; excluded from predictors"

    elif role == "CV grouping":
        preparation = "Standardize street names; use for grouping only"

    elif role == "Split variable":
        preparation = "Use year for chronological train/test split; exclude from predictors"

    else:
        preparation = "Identifier only; exclude from predictors"

    # Add the variable information to the dictionary
    dictionary_rows.append({
        "Variable": variable,
        "Role": role,
        "Type": var_type,
        "Description": description,
        "Missing Count": missing_count,
        "Missing Percent": missing_percent,
        "Unique Values": unique_values,
        "Preparation": preparation
    })

# Convert to a DataFrame
data_dictionary = pd.DataFrame(dictionary_rows)

# Display the dictionary
display(data_dictionary)

# Save the data dictionary
reports_folder = BASE / "reports"
reports_folder.mkdir(parents=True, exist_ok=True)

dictionary_file = reports_folder / "data_dictionary.csv"

data_dictionary.to_csv(dictionary_file, index=False)

print("\nData dictionary saved to:", dictionary_file)

,Variable,Role,Type,Description,Missing Count,Missing Percent,Unique Values,Preparation
0,CRASH_RECORD_ID,Identifier,Categorical,Unique identifier for each reported crash,0,0.00,222301,Identifier only; exclude from predictors
1,CRASH_DATE,Split variable,Datetime,Date and time of the crash,0,0.00,149167,Use year for chronological train/test split; e...
2,STREET_NAME,CV grouping,Categorical,Street name used for grouped cross-validation,0,0.00,1431,Standardize street names; use for grouping only
3,multiple_injuries,Response,Binary,1 if two or more people were injured; 0 otherwise,0,0.00,2,Created from INJURIES_TOTAL >= 2; excluded fro...
4,POSTED_SPEED_LIMIT,Predictor,Numeric,Posted speed limit at the crash location,0,0.00,34,Convert to numeric; median imputation within C...
5,WEATHER_CONDITION,Predictor,Categorical,Weather conditions at the time of the crash,0,0.00,12,Treat missing as Unknown; one-hot encode withi...
6,LIGHTING_CONDITION,Predictor,Categorical,Lighting conditions at the time of the crash,0,0.00,6,Treat missing as Unknown; one-hot encode withi...
7,ROADWAY_SURFACE_COND,Predictor,Categorical,Condition of the roadway surface,0,0.00,7,Treat missing as Unknown; one-hot encode withi...
8,TRAFFIC_CONTROL_DEVICE,Predictor,Categorical,Type of traffic control device present,0,0.00,18,Treat missing as Unknown; one-hot encode withi...
9,ALIGNMENT,Predictor,Categorical,Roadway alignment at the crash location,0,0.00,6,Treat missing as Unknown; one-hot encode withi...



Data dictionary saved to: c:\Users\Samantha\OneDrive\Desktop\School (courses and plans)\Fall 2026\ST 597\Final Project\reports\data_dictionary.csv


### 4. Excluded Variables
List variables excluded due to potential data leakage, including injury-related variables and information collected after the crash. Save the list as a CSV.

In [18]:
# Document variables excluded to prevent data leakage

leakage_variables = [
    "INJURIES_TOTAL",
    "INJURIES_FATAL",
    "INJURIES_INCAPACITATING",
    "INJURIES_NON_INCAPACITATING",
    "INJURIES_REPORTED_NOT_EVIDENT",
    "INJURIES_NO_INDICATION",
    "INJURIES_UNKNOWN",
    "MOST_SEVERE_INJURY",
    "CRASH_TYPE",
    "PRIM_CONTRIBUTORY_CAUSE",
    "SEC_CONTRIBUTORY_CAUSE"
]

# Explain why each variable is excluded
leakage_dictionary = pd.DataFrame({
    "Variable": leakage_variables,
    "Role": "Excluded",
    "Reason": [
        "Used to construct the response variable",
        "Injury severity information determined after the crash",
        "Injury severity information determined after the crash",
        "Injury severity information determined after the crash",
        "Injury severity information determined after the crash",
        "Injury information determined after the crash",
        "Injury information determined after the crash",
        "Injury severity information determined after the crash",
        "Crash classification may contain post-outcome information",
        "Contributing cause determined during investigation",
        "Contributing cause determined during investigation"
    ]
})

# Save the excluded-variable documentation
leakage_file = reports_folder / "excluded_variables.csv"

leakage_dictionary.to_csv(leakage_file, index=False)

display(leakage_dictionary)

print("\nExcluded variables saved to:", leakage_file)

,Variable,Role,Reason
0,INJURIES_TOTAL,Excluded,Used to construct the response variable
1,INJURIES_FATAL,Excluded,Injury severity information determined after t...
2,INJURIES_INCAPACITATING,Excluded,Injury severity information determined after t...
3,INJURIES_NON_INCAPACITATING,Excluded,Injury severity information determined after t...
4,INJURIES_REPORTED_NOT_EVIDENT,Excluded,Injury severity information determined after t...
5,INJURIES_NO_INDICATION,Excluded,Injury information determined after the crash
6,INJURIES_UNKNOWN,Excluded,Injury information determined after the crash
7,MOST_SEVERE_INJURY,Excluded,Injury severity information determined after t...
8,CRASH_TYPE,Excluded,Crash classification may contain post-outcome ...
9,PRIM_CONTRIBUTORY_CAUSE,Excluded,Contributing cause determined during investiga...



Excluded variables saved to: c:\Users\Samantha\OneDrive\Desktop\School (courses and plans)\Fall 2026\ST 597\Final Project\reports\excluded_variables.csv


### 5. Grouped Cross-Validation
Create 5 folds grouped by `STREET_NAME` to keep crashes from the same street together. Check fold sizes and positive rates, verify no overlapping streets, and save fold assignments.

In [20]:
# Set up grouped 5-fold cross-validation

from sklearn.model_selection import GroupKFold

# Set the random seed for later modeling
SEED = 597

# Use street names to group crashes
groups = (
    train["STREET_NAME"]
    .fillna("UNKNOWN_STREET")
    .astype(str)
    .str.strip()
    .str.upper()
)

# Create the grouped cross-validation object
cv = GroupKFold(n_splits=5)

# Create a column to record each observation's validation fold
train["CV_FOLD"] = -1

# Store information about each fold
fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(train, train["multiple_injuries"], groups=groups),
    start=1
):

    # Record which validation fold each observation belongs to
    train.iloc[
        val_idx,
        train.columns.get_loc("CV_FOLD")
    ] = fold

    # Check that no street appears in both portions
    train_streets = set(groups.iloc[train_idx])
    val_streets = set(groups.iloc[val_idx])

    assert train_streets.isdisjoint(val_streets)

    # Calculate fold information
    fold_results.append({
        "Fold": fold,
        "Training observations": len(train_idx),
        "Validation observations": len(val_idx),
        "Validation positive rate": round(
            train.iloc[val_idx]["multiple_injuries"].mean() * 100, 2
        ),
        "Validation streets": len(val_streets)
    })

# Create a summary table
fold_summary = pd.DataFrame(fold_results)

display(fold_summary)

# Verify every observation was assigned a fold
assert (train["CV_FOLD"] >= 1).all()

# Save fold assignments separately from model predictors
fold_file = processed_folder / "train_fold_assignments.csv"

train[["CRASH_RECORD_ID", "CV_FOLD"]].to_csv(
    fold_file,
    index=False
)

print("\nFold assignments saved to:", fold_file)
print("Grouped 5-fold cross-validation verified successfully!")

,Fold,Training observations,Validation observations,Validation positive rate,Validation streets
0,1,177840,44461,4.00,287
1,2,177841,44460,4.17,286
2,3,177841,44460,3.94,286
3,4,177841,44460,3.92,286
4,5,177841,44460,4.44,286



Fold assignments saved to: c:\Users\Samantha\OneDrive\Desktop\School (courses and plans)\Fall 2026\ST 597\Final Project\data\processed\train_fold_assignments.csv
Grouped 5-fold cross-validation verified successfully!


### 6. Baseline Model
Fit a majority-class baseline using `DummyClassifier` across the 5 folds. Calculate accuracy, precision, recall, F1, average precision, and ROC-AUC; save results for comparison with future models.

In [21]:
# Calculate baseline performance using grouped 5-fold cross-validation

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

# Set the random seed
SEED = 597
np.random.seed(SEED)

# Create a list to store the results from each fold
baseline_results = []

# Use the five folds created in the previous section
for fold in range(1, 6):

    # Separate the training and validation observations
    fold_train = train[train["CV_FOLD"] != fold].copy()
    fold_val = train[train["CV_FOLD"] == fold].copy()

    # Define the response variable
    y_train = fold_train["multiple_injuries"]
    y_val = fold_val["multiple_injuries"]

    # DummyClassifier requires an X variable, but ignores its values
    X_train = np.zeros((len(fold_train), 1))
    X_val = np.zeros((len(fold_val), 1))

    # Fit a baseline model that always predicts the majority class
    baseline = DummyClassifier(
        strategy="most_frequent",
        random_state=SEED
    )

    baseline.fit(X_train, y_train)

    # Generate class predictions and predicted probabilities
    y_pred = baseline.predict(X_val)

    # Extract the predicted probability of the positive class
    if 1 in baseline.classes_:
        positive_index = list(baseline.classes_).index(1)
        y_prob = baseline.predict_proba(X_val)[:, positive_index]
    else:
        y_prob = np.zeros(len(y_val))

    # Calculate the baseline evaluation metrics
    baseline_results.append({
        "Fold": fold,
        "Training Observations": len(fold_train),
        "Validation Observations": len(fold_val),
        "Validation Positive Rate": y_val.mean(),
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1": f1_score(y_val, y_pred, zero_division=0),
        "Average Precision": (
            average_precision_score(y_val, y_prob)
            if y_val.nunique() == 2 else np.nan
        ),
        "ROC-AUC": (
            roc_auc_score(y_val, y_prob)
            if y_val.nunique() == 2 else np.nan
        )
    })

# Convert results to a DataFrame
baseline_results = pd.DataFrame(baseline_results)

# Display the results for all five folds
print("Baseline results by fold:")
display(baseline_results.round(4))

# Calculate the average performance across folds
baseline_summary = baseline_results[
    [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Average Precision",
        "ROC-AUC"
    ]
].agg(["mean", "std"])

print("\nBaseline performance summary:")
display(baseline_summary.round(4))

# Save the results
reports_folder = BASE / "reports"
reports_folder.mkdir(parents=True, exist_ok=True)

baseline_results.to_csv(
    reports_folder / "baseline_grouped_cv.csv",
    index=False
)

baseline_summary.to_csv(
    reports_folder / "baseline_summary.csv"
)

print("\nBaseline results saved successfully!")

Baseline results by fold:


,Fold,Training Observations,Validation Observations,Validation Positive Rate,Accuracy,Precision,Recall,F1,Average Precision,ROC-AUC
0,1,177840,44461,0.0400,0.9600,0.0,0.0,0.0,0.0400,0.5
1,2,177841,44460,0.0417,0.9583,0.0,0.0,0.0,0.0417,0.5
2,3,177841,44460,0.0394,0.9606,0.0,0.0,0.0,0.0394,0.5
3,4,177841,44460,0.0392,0.9608,0.0,0.0,0.0,0.0392,0.5
4,5,177841,44460,0.0444,0.9556,0.0,0.0,0.0,0.0444,0.5



Baseline performance summary:


,Accuracy,Precision,Recall,F1,Average Precision,ROC-AUC
mean,0.9591,0.0,0.0,0.0,0.0409,0.5
std,0.0022,0.0,0.0,0.0,0.0022,0.0



Baseline results saved successfully!


### 7. Preprocessing & Model Tuning
Define preprocessing for numeric and categorical variables and set up pipelines for Logistic Regression, Random Forest, and Gradient Boosting. Set hyperparameter grids using seed 597 and save configurations without fitting the models.

In [22]:
# Define preprocessing and hyperparameter grids for the models

import json

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Set the random seed
SEED = 597

# Define the predictor variables
numeric_features = [
    "POSTED_SPEED_LIMIT"
]

categorical_features = [
    "WEATHER_CONDITION",
    "LIGHTING_CONDITION",
    "ROADWAY_SURFACE_COND",
    "TRAFFIC_CONTROL_DEVICE",
    "ALIGNMENT",
    "ROAD_DEFECT",
    "INTERSECTION_RELATED_I",
    "STREET_DIRECTION",
    "TRAFFICWAY_TYPE",
    "CRASH_HOUR",
    "CRASH_DAY_OF_WEEK",
    "CRASH_MONTH",
    "WORK_ZONE_I",
    "WORK_ZONE_TYPE",
    "WORKERS_PRESENT_I"
]

predictors = numeric_features + categorical_features

# Verify all predictors are present
missing_predictors = [
    col for col in predictors
    if col not in train.columns
]

if missing_predictors:
    raise ValueError(f"Missing predictors: {missing_predictors}")

print("Number of predictors:", len(predictors))


# Define preprocessing for numeric variables
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Define preprocessing for categorical variables
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


# Define the three model pipelines
models = {

    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            solver="saga",
            max_iter=2000,
            random_state=SEED
        ))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            random_state=SEED,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", HistGradientBoostingClassifier(
            random_state=SEED
        ))
    ])
}


# Define hyperparameter grids
param_grids = {

    "Logistic Regression": {
        "model__penalty": ["l1", "l2"],
        "model__C": [0.01, 0.1, 1, 10],
        "model__class_weight": [None, "balanced"]
    },

    "Random Forest": {
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 5, 10],
        "model__class_weight": [None, "balanced"]
    },

    "Gradient Boosting": {
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3],
        "model__max_iter": [100, 200]
    }
}


# Count the combinations in each grid
from sklearn.model_selection import ParameterGrid

for model_name, grid in param_grids.items():

    n_combinations = len(ParameterGrid(grid))

    print(f"{model_name}: {n_combinations} combinations")


# Save the proposed model configurations
model_config = {
    "random_seed": SEED,
    "primary_metric": "average_precision",
    "cross_validation": "5-fold GroupKFold using STREET_NAME",
    "numeric_predictors": numeric_features,
    "categorical_predictors": categorical_features,
    "numeric_preprocessing": [
        "Median imputation",
        "Standard scaling"
    ],
    "categorical_preprocessing": [
        "Constant imputation with Unknown",
        "One-hot encoding with unknown categories ignored"
    ],
    "hyperparameter_grids": param_grids
}

reports_folder = BASE / "reports"
reports_folder.mkdir(parents=True, exist_ok=True)

config_file = reports_folder / "model_protocol_config.json"

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=4)

print("\nModel configurations saved to:", config_file)
print("No predictive models have been fitted.")

Number of predictors: 16
Logistic Regression: 16 combinations
Random Forest: 18 combinations
Gradient Boosting: 8 combinations

Model configurations saved to: c:\Users\Samantha\OneDrive\Desktop\School (courses and plans)\Fall 2026\ST 597\Final Project\reports\model_protocol_config.json
No predictive models have been fitted.
